# Climate dashboard: data download, organised by GCOS Essential Climate Variable

Downloads every file the dashboard needs that Claude can't fetch itself, checks it, previews it, and bundles it into one zip to drag into the chat. Re-run it any time to refresh; it never replaces a good file with a broken download.

| GCOS ECV | Files | Source |
|---|---|---|
| Carbon dioxide, methane and other greenhouse gases | `co2_mm_gl.csv`, `ch4_mm_gl.csv`, `n2o_mm_gl.csv`, `co2_mm_mlo.csv`, `AGGI_Table.csv` (+ optional annual means and growth rates) | NOAA Global Monitoring Laboratory |
| Surface temperature | `GLB.Ts+dSST.csv`, `NH.Ts+dSST.csv`, `SH.Ts+dSST.csv`, `ZonAnn.Ts+dSST.csv` | NASA GISS, GISTEMP v4 |
| Ocean colour | `chl_*.nc` grids + `chl_global_monthly.csv`, `chl_region_monthly.csv` | NOAA CoastWatch ERDDAP, S-NPP VIIRS science-quality monthly chlorophyll-a |
| Sea surface temperature (ENSO) | `oni.ascii.txt`, `RONI.ascii.txt` (off by default: you already have these) | NOAA Climate Prediction Center |

Our World in Data's emissions file lives on GitHub, so Claude fetches that one directly; it isn't here.

**How to run:** in [Google Colab](https://colab.research.google.com) choose **File → Upload notebook**, then **Runtime → Run all**; the last cell downloads `climate_data.zip`. In Jupyter or VS Code, run all cells and the files land in `climate_data/` next to the notebook. The chlorophyll step needs `xarray` (already in Colab; elsewhere `pip install xarray netCDF4`) and takes a few minutes.

*Credits:* NOAA Global Monitoring Laboratory ([gml.noaa.gov/ccgg/trends](https://gml.noaa.gov/ccgg/trends/), [AGGI](https://gml.noaa.gov/aggi/aggi.html)); Mauna Loa CO₂: Dr. Xin Lan, NOAA/GML, and Dr. Ralph Keeling, Scripps Institution of Oceanography. GISTEMP Team, NASA GISS ([data.giss.nasa.gov/gistemp](https://data.giss.nasa.gov/gistemp/)). NOAA NESDIS CoastWatch / STAR ocean colour ([coastwatch.noaa.gov/erddap](https://coastwatch.noaa.gov/erddap/)).

## 1. Settings

`get=True` downloads a file; set it to `False` to skip it. The chlorophyll region defaults to the Santa Barbara Channel; change the box to any region you like.

In [ ]:
from pathlib import Path

OUT_DIR = Path("climate_data")

GML = "https://gml.noaa.gov/webdata/ccgg/trends"
GML_FTP = "https://gml.noaa.gov/aftp/products/trends"
GISS = "https://data.giss.nasa.gov/gistemp/tabledata_v4"

FILES = [
    # ── Carbon dioxide, methane and other greenhouse gases (Atmosphere) ──
    dict(get=True,  name="co2_mm_gl.csv",  ecv="Greenhouse gases", what="Global CO2, monthly (ppm)",
         urls=[f"{GML}/co2/co2_mm_gl.csv", f"{GML_FTP}/co2/co2_mm_gl.csv"]),
    dict(get=True,  name="ch4_mm_gl.csv",  ecv="Greenhouse gases", what="Global methane, monthly (ppb)",
         urls=[f"{GML}/ch4/ch4_mm_gl.csv"]),
    dict(get=True,  name="n2o_mm_gl.csv",  ecv="Greenhouse gases", what="Global N2O, monthly (ppb)",
         urls=[f"{GML}/n2o/n2o_mm_gl.csv"]),
    dict(get=True,  name="co2_mm_mlo.csv", ecv="Greenhouse gases", what="Mauna Loa CO2, monthly, from 1958 (ppm)",
         urls=[f"{GML}/co2/co2_mm_mlo.csv"]),
    dict(get=True,  name="AGGI_Table.csv", ecv="Greenhouse gases", what="Radiative forcing by gas, annual (W m-2)",
         urls=["https://gml.noaa.gov/aggi/AGGI_Table.csv"]),
    dict(get=False, name="co2_annmean_gl.csv", ecv="Greenhouse gases", what="Global CO2, annual mean",
         urls=[f"{GML}/co2/co2_annmean_gl.csv", f"{GML_FTP}/co2/co2_annmean_gl.csv"]),
    dict(get=False, name="ch4_annmean_gl.csv", ecv="Greenhouse gases", what="Global methane, annual mean",
         urls=[f"{GML}/ch4/ch4_annmean_gl.csv"]),
    dict(get=False, name="n2o_annmean_gl.csv", ecv="Greenhouse gases", what="Global N2O, annual mean",
         urls=[f"{GML}/n2o/n2o_annmean_gl.csv"]),
    dict(get=False, name="co2_gr_gl.csv", ecv="Greenhouse gases", what="Global CO2, annual growth rate",
         urls=[f"{GML}/co2/co2_gr_gl.csv", f"{GML_FTP}/co2/co2_gr_gl.csv"]),
    dict(get=False, name="ch4_gr_gl.csv", ecv="Greenhouse gases", what="Global methane, annual growth rate",
         urls=[f"{GML}/ch4/ch4_gr_gl.csv"]),
    dict(get=False, name="n2o_gr_gl.csv", ecv="Greenhouse gases", what="Global N2O, annual growth rate",
         urls=[f"{GML}/n2o/n2o_gr_gl.csv"]),

    # ── Surface temperature (Atmosphere) ──
    dict(get=True, name="GLB.Ts+dSST.csv", ecv="Surface temperature", what="Global temperature anomaly, monthly (°C vs 1951-1980)",
         urls=[f"{GISS}/GLB.Ts+dSST.csv"]),
    dict(get=True, name="NH.Ts+dSST.csv", ecv="Surface temperature", what="Northern Hemisphere anomaly, monthly",
         urls=[f"{GISS}/NH.Ts+dSST.csv"]),
    dict(get=True, name="SH.Ts+dSST.csv", ecv="Surface temperature", what="Southern Hemisphere anomaly, monthly",
         urls=[f"{GISS}/SH.Ts+dSST.csv"]),
    dict(get=True, name="ZonAnn.Ts+dSST.csv", ecv="Surface temperature", what="Latitude-band anomalies, annual",
         urls=[f"{GISS}/ZonAnn.Ts+dSST.csv"]),

    # ── Sea surface temperature: ENSO (you already have these; switch on to keep them refreshed) ──
    dict(get=False, name="oni.ascii.txt", ecv="Sea surface temperature", what="Oceanic Niño Index (ONI)",
         urls=["https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"]),
    dict(get=False, name="RONI.ascii.txt", ecv="Sea surface temperature", what="Relative Oceanic Niño Index (RONI)",
         urls=["https://www.cpc.ncep.noaa.gov/data/indices/RONI.ascii.txt"]),
]

# ── Ocean colour: chlorophyll-a from ERDDAP (subsets, so no huge downloads) ──
CHL = dict(
    get=True,
    sources=[  # tried in order: (ERDDAP server, dataset ID)
        ("https://coastwatch.noaa.gov/erddap", "noaacwNPPVIIRSSQchlaMonthly"),
        ("https://polarwatch.noaa.gov/erddap", "nesdisVHNSQchlaMonthly"),
    ],
    variable=None,          # None = find the chlorophyll variable automatically
    record_deg=2.0,         # whole record, global grid every ~2 degrees
    recent_deg=1.0,         # latest months, global grid every ~1 degree
    recent_months=12,
    region_name="Santa Barbara Channel",
    region_lat=(33.9, 34.5),       # south, north (degrees)
    region_lon=(-120.5, -119.3),   # west, east (degrees, -180 to 180)
    band_lat=(-50, 50),     # band for the "global" average; avoids polar-night gaps
)

## 2. Download the direct files

Each file is checked before it's saved: an empty reply or an error page never replaces a good copy you already have.

In [ ]:
from urllib.request import Request, urlopen
from urllib.error import HTTPError

HEADERS = {"User-Agent": "Mozilla/5.0 (personal climate dashboard; python urllib)"}

def http_get(url, timeout=120):
    """Fetch a URL and return its bytes (raises on failure)."""
    with urlopen(Request(url, headers=HEADERS), timeout=timeout) as resp:
        return resp.read()

def explain(err):
    text = str(getattr(err, "reason", err))
    if "CERTIFICATE_VERIFY_FAILED" in text:
        return ("SSL certificate error. On a Mac, open Finder > Applications > Python 3.x and "
                "double-click 'Install Certificates.command', then re-run.")
    if isinstance(err, HTTPError):
        return f"HTTP {err.code}"
    return text

def looks_like_data(data):
    head = data[:2000].lower()
    return len(data) > 200 and b"<html" not in head and b"<!doctype" not in head

def download(item):
    for url in item["urls"]:
        try:
            data = http_get(url)
        except Exception as e:
            print(f"  ✗ {item['name']}: {explain(e)} ({url})")
            continue
        if not looks_like_data(data):
            print(f"  ✗ {item['name']}: reply from {url} isn't data; kept any previous copy")
            continue
        (OUT_DIR / item["name"]).write_bytes(data)
        print(f"  ✓ {item['name']:<20} {len(data) / 1024:>6.0f} KB   {item['what']}")
        return True
    return False

OUT_DIR.mkdir(exist_ok=True)
print(f"Saving to: {OUT_DIR.resolve()}\n")
wanted = [f for f in FILES if f["get"]]
failed = [f["name"] for f in wanted if not download(f)]
print(f"\n{len(wanted) - len(failed)} of {len(wanted)} files downloaded.")
if failed:
    print("Not downloaded:", ", ".join(failed), "\nYou can open those links in a browser and save the page instead.")

## 3. Ocean colour: chlorophyll-a subsets from ERDDAP

ERDDAP lets us ask for a thinned-out grid straight from a URL, so three small netCDF files replace terabytes of imagery:

- the whole record on a ~2° global grid (for the global series and animated maps),
- the latest months on a ~1° global grid (for sharper maps),
- the region box at full 4 km resolution (for the regional series).

The dataset's layout (variable and axis names, sizes) is read from the server first, so nothing is hard-coded. This step can take a few minutes.

In [ ]:
import re
from urllib.parse import quote

def erddap_url(server, dataset, ext, query=""):
    return f"{server}/griddap/{dataset}.{ext}" + (f"?{quote(query, safe='(),')}" if query else "")

def erddap_grids(server, dataset):
    """Read the dataset structure (.dds): {variable: [(dimension, size), ...]}."""
    dds = http_get(erddap_url(server, dataset, "dds")).decode()
    grids = {}
    for m in re.finditer(r"ARRAY:\s+\w+\s+(\w+)((?:\[[^\]]+\])+);", dds):
        grids[m.group(1)] = [(d, int(n)) for d, n in re.findall(r"\[\s*(\w+)\s*=\s*(\d+)\s*\]", m.group(2))]
    if not grids:
        raise ValueError("no gridded variables found")
    return grids

def erddap_axis(server, dataset, axis):
    """All values of one axis (first two CSV rows are the name and the units)."""
    lines = http_get(erddap_url(server, dataset, "csv", axis)).decode().strip().splitlines()
    return [v.strip() for v in lines[2:]]

def stride_for(values, degrees):
    return max(1, round(degrees / abs(values[1] - values[0])))

def index_range(values, lo, hi):
    idx = [i for i, v in enumerate(values) if lo <= v <= hi]
    if not idx:
        raise ValueError(f"no grid points between {lo} and {hi}")
    return min(idx), max(idx)

def grid_query(var, dims, sel):
    return var + "".join("[{}:{}:{}]".format(*sel.get(d, (0, 1, n - 1))) for d, n in dims)

def get_chlorophyll():
    for server, dataset in CHL["sources"]:
        try:
            grids = erddap_grids(server, dataset)
            break
        except Exception as e:
            print(f"  ✗ {dataset} at {server}: {explain(e)}")
    else:
        print("No chlorophyll server reachable; skipping this step.")
        return None

    var = CHL["variable"] or ("chlor_a" if "chlor_a" in grids else next(v for v in grids if "chl" in v.lower()))
    dims = grids[var]
    size = dict(dims)
    t = next(d for d in size if d.lower() == "time")
    la = next(d for d in size if d.lower().startswith("lat"))
    lo = next(d for d in size if d.lower().startswith("lon"))
    lat = [float(v) for v in erddap_axis(server, dataset, la)]
    lon = [float(v) for v in erddap_axis(server, dataset, lo)]
    times = erddap_axis(server, dataset, t)
    nt = size[t]
    print(f"  {dataset}: '{var}', {nt} months ({times[0][:7]} to {times[-1][:7]}), {size[la]} x {size[lo]} grid\n")

    west, east = CHL["region_lon"]
    if max(lon) > 180:                       # this server uses 0-360 longitudes
        west, east = west % 360, east % 360
    r_lat, r_lon = index_range(lat, *CHL["region_lat"]), index_range(lon, west, east)
    s2, s1 = stride_for(lat, CHL["record_deg"]), stride_for(lat, CHL["recent_deg"])
    k = min(CHL["recent_months"], nt)
    jobs = [
        (f"chl_global_{CHL['record_deg']:g}deg_all.nc",
         {t: (0, 1, nt - 1), la: (0, s2, size[la] - 1), lo: (0, s2, size[lo] - 1)}, "whole record, coarse global grid"),
        (f"chl_global_{CHL['recent_deg']:g}deg_last{k}.nc",
         {t: (nt - k, 1, nt - 1), la: (0, s1, size[la] - 1), lo: (0, s1, size[lo] - 1)}, f"last {k} months, finer global grid"),
        ("chl_region.nc",
         {t: (0, 1, nt - 1), la: (r_lat[0], 1, r_lat[1]), lo: (r_lon[0], 1, r_lon[1])}, f"{CHL['region_name']}, full resolution"),
    ]
    saved = []
    for fname, sel, what in jobs:
        try:
            data = http_get(erddap_url(server, dataset, "nc", grid_query(var, dims, sel)), timeout=900)
        except Exception as e:
            print(f"  ✗ {fname}: {explain(e)}")
            continue
        if not (data[:3] == b"CDF" or data[:4] == b"\x89HDF"):
            print(f"  ✗ {fname}: reply isn't a netCDF file; kept any previous copy")
            continue
        (OUT_DIR / fname).write_bytes(data)
        print(f"  ✓ {fname:<28} {len(data) / 1024**2:>5.1f} MB   {what}")
        saved.append(fname)
    return dict(server=server, dataset=dataset, var=var, files=saved)

chl_info = get_chlorophyll() if CHL["get"] else None

In [ ]:
# Monthly chlorophyll series from the grids: area-weighted geometric means
# (chlorophyll is roughly log-normal, so averages are taken in log space).
import numpy as np
import pandas as pd
try:
    import xarray as xr
except ImportError:
    xr = None

def chl_monthly(path, var, lat_band=None):
    with xr.open_dataset(path) as ds:
        da = ds[var]
        extra = [d for d in da.dims if da.sizes[d] == 1 and d != "time" and not d.lower().startswith(("lat", "lon"))]
        da = da.squeeze(extra, drop=True)
        da = da.where((da > 0) & (da < 100))                 # land, cloud and fill values -> NaN
        la = next(d for d in da.dims if d.lower().startswith("lat"))
        lo = next(d for d in da.dims if d.lower().startswith("lon"))
        if lat_band:
            da = da.where((da[la] >= lat_band[0]) & (da[la] <= lat_band[1]), drop=True)
        w = (np.cos(np.deg2rad(da[la])) * xr.ones_like(da)).where(da.notnull())
        mean_log = (np.log10(da) * w).sum((la, lo)) / w.sum((la, lo))
        return pd.DataFrame({
            "date": pd.to_datetime(da["time"].values).strftime("%Y-%m"),
            "chl_mg_m3": (10 ** mean_log).values.round(4),
            "valid_share": da.notnull().mean((la, lo)).values.round(3),
        })

def latest(s):
    ok = s.dropna(subset=["chl_mg_m3"])
    if ok.empty:
        return "(no valid pixels: is the box over land, or always cloudy?)"
    return f"{ok['date'].iloc[-1]}: {ok['chl_mg_m3'].iloc[-1]:.3f} mg m-3"

if chl_info and xr is None:
    print("Install xarray to build the chlorophyll series:  pip install xarray netCDF4")
elif chl_info:
    var = chl_info["var"]
    g = OUT_DIR / f"chl_global_{CHL['record_deg']:g}deg_all.nc"
    if g.exists():
        s = chl_monthly(g, var, CHL["band_lat"])
        s.insert(1, "band", f"{CHL['band_lat'][0]} to {CHL['band_lat'][1]} lat")
        s.to_csv(OUT_DIR / "chl_global_monthly.csv", index=False)
        print(f"  ✓ chl_global_monthly.csv   {len(s)} months, latest {latest(s)}")
    r = OUT_DIR / "chl_region.nc"
    if r.exists():
        s = chl_monthly(r, var)
        s.insert(1, "region", CHL["region_name"])
        s.to_csv(OUT_DIR / "chl_region_monthly.csv", index=False)
        print(f"  ✓ chl_region_monthly.csv   {len(s)} months, latest {latest(s)}")

## 4. Quick look

A sanity check: does each file read cleanly, and how current is it? This only reads the files.

In [ ]:
import csv

MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

def read_noaa(path):
    """NOAA GML file: '#' comment header, then year, month, ..., average, ..."""
    df = pd.read_csv(path, comment="#")
    df.columns = [c.strip().lower() for c in df.columns]
    value_col = "average" if "average" in df.columns else ("mean" if "mean" in df.columns else df.columns[-1])
    day = df["month"] if "month" in df.columns else 7
    df["date"] = pd.to_datetime(dict(year=df["year"], month=day, day=15))
    df["value"] = pd.to_numeric(df[value_col], errors="coerce")
    return df.loc[df["value"] > 0, ["date", "value"]]       # NOAA marks missing values as negative

def read_gistemp(path):
    """NASA GISTEMP monthly table: a title line, then Year, Jan..Dec, ...; '***' = not yet available."""
    df = pd.read_csv(path, skiprows=1, na_values=["***", "****"])
    long = df.melt(id_vars="Year", value_vars=MONTHS, var_name="month", value_name="value")
    long["value"] = pd.to_numeric(long["value"], errors="coerce")
    long = long.dropna(subset=["value"])
    long["date"] = pd.to_datetime(long["Year"].astype(int).astype(str) + "-" + long["month"] + "-15", format="%Y-%b-%d")
    return long.sort_values("date")[["date", "value"]].reset_index(drop=True)

def read_zonal(path):
    df = pd.read_csv(path, na_values=["***", "****"])
    df["date"] = pd.to_datetime(df["Year"].astype(int).astype(str) + "-07-01")
    return df.rename(columns={"Glob": "value"})[["date", "value"]]

def read_aggi(path):
    """AGGI table: header rows of any shape, then one row per year; returns year + total forcing."""
    rows = [r for r in csv.reader(open(path, encoding="utf-8", errors="replace")) if r]
    data = [r for r in rows if re.fullmatch(r"\d{4}", r[0].strip())]
    header = [r for r in rows if not re.fullmatch(r"\d{4}", r[0].strip())]
    width = max(len(r) for r in data)
    labels = [" ".join(h[i] for h in header if i < len(h)).lower() for i in range(width)]
    col = next(i for i in range(1, width) if "total" in labels[i])
    return pd.DataFrame({"date": pd.to_datetime([r[0].strip() + "-07-01" for r in data]),
                         "value": pd.to_numeric([r[col] for r in data], errors="coerce")}).dropna()

def read_chl_csv(path):
    df = pd.read_csv(path)
    return pd.DataFrame({"date": pd.to_datetime(df["date"] + "-15"), "value": df["chl_mg_m3"]}).dropna()

def read_enso(path):
    df = pd.read_csv(path, sep=r"\s+")
    df.columns = [c.upper() for c in df.columns]
    val = "ANOM" if "ANOM" in df.columns else df.columns[-1]
    return pd.DataFrame({"date": pd.to_datetime(df["YR"].astype(str) + "-07-01"), "value": df[val]})

def reader_for(name):
    if name.endswith(".nc"):            return None
    if name.startswith("AGGI"):         return read_aggi
    if name.startswith("ZonAnn"):       return read_zonal
    if name.endswith("Ts+dSST.csv"):    return read_gistemp
    if name.startswith("chl_"):         return read_chl_csv
    if name.endswith(".ascii.txt"):     return read_enso
    return read_noaa

data, rows = {}, []
for path in sorted(OUT_DIR.iterdir()):
    reader = reader_for(path.name)
    if reader is None:
        rows.append([path.name, "netCDF grid", "", "", "", f"{path.stat().st_size / 1024**2:.1f} MB"])
        continue
    try:
        df = reader(path)
        if df.empty:
            rows.append([path.name, "no valid values", "", "", "", ""])
            continue
        data[path.name] = df
        rows.append([path.name, f"{len(df):,} rows", df["date"].min().strftime("%Y-%m"),
                     df["date"].max().strftime("%Y-%m"), round(float(df["value"].iloc[-1]), 3), ""])
    except Exception as e:
        rows.append([path.name, f"could not read: {e}", "", "", "", ""])

summary = pd.DataFrame(rows, columns=["File", "Rows", "From", "To", "Latest value", "Size"])
summary

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter
try:
    import cmcrameri.cm as cmc
    CHL_CMAP = cmc.batlow
except ImportError:
    CHL_CMAP = "viridis"

INK, INK_2, GRID, SURFACE, LINE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb", "#2a78d6"
PANELS = [
    ("co2_mm_gl.csv", "Global CO₂ · ppm"),
    ("ch4_mm_gl.csv", "Global methane · ppb"),
    ("n2o_mm_gl.csv", "Global N₂O · ppb"),
    ("GLB.Ts+dSST.csv", "Global temperature · °C vs 1951–1980"),
    ("AGGI_Table.csv", "Forcing, all long-lived gases · W m⁻²"),
    ("ZonAnn.Ts+dSST.csv", "Global temperature, annual · °C"),
    ("chl_global_monthly.csv", "Chlorophyll, 50°S–50°N · mg m⁻³"),
    ("chl_region_monthly.csv", f"Chlorophyll, {CHL['region_name']} · mg m⁻³"),
]

def style(ax, title):
    ax.set_facecolor(SURFACE)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_2, labelsize=8.5, length=0)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_title(title, loc="left", color=INK, fontsize=10.5, pad=8)

fig, axes = plt.subplots(3, 3, figsize=(13, 9.5), facecolor=SURFACE)
for ax, (fname, title) in zip(axes.flat, PANELS):
    style(ax, title)
    if fname not in data or data[fname].empty:
        ax.text(0.5, 0.5, f"{fname} not available", transform=ax.transAxes, ha="center", va="center", color=INK_2)
        continue
    df = data[fname]
    if "Temperature" in title or "temperature" in title:
        ax.axhline(0, color=INK_2, linewidth=0.8)
    if fname.startswith("chl_") and df["value"].max() / df["value"].min() > 5:
        ax.set_yscale("log")
        ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
        ax.yaxis.set_minor_formatter(NullFormatter())
    locator = mdates.AutoDateLocator(minticks=3, maxticks=6)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    thin = fname == "GLB.Ts+dSST.csv"
    ax.plot(df["date"], df["value"], color=LINE, linewidth=0.8 if thin else 1.2)
    last = df.iloc[-1]
    ax.annotate(f"{last['value']:.2f}  ({last['date']:%b %Y})", (last["date"], last["value"]),
                xytext=(-4, 8), textcoords="offset points", ha="right", color=INK, fontsize=8.5)

ax = axes.flat[8]
recent = sorted(OUT_DIR.glob("chl_global_*deg_last*.nc"))
if recent and xr is not None and chl_info:
    with xr.open_dataset(recent[0]) as ds:
        da = ds[chl_info["var"]]
        da = da.squeeze([d for d in da.dims if da.sizes[d] == 1 and d != "time"], drop=True).isel(time=-1)
        da = da.where((da > 0) & (da < 100))
        la = next(d for d in da.dims if d.lower().startswith("lat"))
        lo = next(d for d in da.dims if d.lower().startswith("lon"))
        mesh = ax.pcolormesh(da[lo], da[la], np.log10(da), cmap=CHL_CMAP, vmin=-1.7, vmax=1.0, shading="auto")
        ax.set_title(f"Chlorophyll, {pd.Timestamp(da['time'].values):%b %Y} · log₁₀ mg m⁻³", loc="left", color=INK, fontsize=10.5, pad=8)
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
        fig.colorbar(mesh, ax=ax, orientation="horizontal", fraction=0.05, pad=0.04).ax.tick_params(labelsize=8, colors=INK_2)
else:
    style(ax, "Latest chlorophyll map")
    ax.text(0.5, 0.5, "chlorophyll grid not available", transform=ax.transAxes, ha="center", va="center", color=INK_2)

fig.suptitle("Downloaded data at a glance", x=0.01, ha="left", color=INK, fontsize=13)
fig.tight_layout()
plt.show()

## 5. Send the files

In Colab this bundles everything into `climate_data.zip` and starts the download. Locally it tells you where the folder is. Then drag the zip (or the files) into the chat.

In [ ]:
import shutil

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

zip_path = shutil.make_archive("climate_data", "zip", OUT_DIR)
size_mb = Path(zip_path).stat().st_size / 1024**2
if colab_files:
    colab_files.download(zip_path)
    print(f"Downloading climate_data.zip ({size_mb:.1f} MB). Drag it into the chat.")
else:
    print(f"Zip ready: {Path(zip_path).resolve()}  ({size_mb:.1f} MB)\nFiles:")
    for p in sorted(OUT_DIR.iterdir()):
        print("  ", p.name)